### Q1. YouTube 트렌딩 영상 데이터셋에서, 날짜순으로 정렬했을 때, 전날에도 트렌딩에 있었고 dislike 수가 전일보다 증가한 날이 가장 길게 이어진 영상의 channelTitle을 출력하라

In [33]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("thedevastator/youtube-trending-videos-dataset")
df = pd.read_csv(path + "/youtube.csv")

print(df)

         index     video_id trending_date  \
0            0  2kyS6SvSYSE      17.14.11   
1            1  1ZAPwfrtAFY      17.14.11   
2            2  5qpjK5DgCt4      17.14.11   
3            3  puqaWrEC7tY      17.14.11   
4            4  d380meD0W0M      17.14.11   
...        ...          ...           ...   
161465  161465  sGolxsMSGfQ      18.14.06   
161466  161466  8HNuRNi8t70      18.14.06   
161467  161467  GWlKEM3m2EE      18.14.06   
161468  161468  lbMKLzQ4cNQ      18.14.06   
161469  161469  POTgw38-m58      18.14.06   

                                                    title  \
0                      WE WANT TO TALK ABOUT OUR MARRIAGE   
1       The Trump Presidency: Last Week Tonight with J...   
2       Racist Superman | Rudy Mancuso, King Bach & Le...   
3                        Nickelback Lyrics: Real or Fake?   
4                                I Dare You: GOING BALD!?   
...                                                   ...   
161465                       HOW

shift 만 사용 (X)

In [ ]:
df_sort = df.sort_values(by=['title','trending_date'],ascending=True)
df_sort['trending_date'] = pd.to_datetime(df_sort['trending_date'],format='%y.%d.%m')

df_sort['pre_trending'] = df_sort['trending_date'].shift(1)
df_sort['pre_dislikes'] = df_sort['dislikes'].shift(1)
df_sort['pre_title'] = df_sort['title'].shift(1)
trend_mask = (df_sort['trending_date'] - df_sort['pre_trending'])==pd.Timedelta(1,"D")
dislikes_mask = (df_sort['dislikes']>df_sort['pre_dislikes'])
title_mask = df_sort['pre_title'] == df_sort['title']

answer_df = df_sort[trend_mask & dislikes_mask & title_mask]

groupby 사용
mask를 따로 만들기보다 새로운 column 추가하는 형태의 작업 필수

True를 카운트 하는 방법 </br>

장점: 연속된 길이를 쉽게 카운트 할 수 있다. </br>
단점: 연속이 끊어질 때 카운트를 끊어야 한다 </br>
</br>
</br>
False를 카운트 하는 방법 </br>
장점: 같은 그룹으로 묶기 좋다. </br>
단점: 가장 큰 그룹을 찾고 그 그룹의 channeltitle을 출력하는 노고가 필요하다.



In [ ]:
df['trending_date'] = pd.to_datetime(df['trending_date'], format='%y.%d.%m')
df_sort = df.sort_values(by=['title','trending_date'], ascending=True)

df_sort['trending_diff'] = df_sort.groupby('title')['trending_date'].diff()

# groupby 사용 용례
# print(df_sort['dislikes_False'].groupby(df_sort['channel_title']).size())
# print(df_sort.groupby(df_sort['channel_title'])['dislikes_False'].size())
# groupby에 as_index개념 활용하면 골치가 반으로 줄어듬

#어제와 연속하면서 dislikes 증가
df_sort['dislikes_up'] = (df_sort['trending_diff'] == pd.Timedelta(days=1)) & (df_sort.groupby('title')['dislikes'].diff() > 0)

# dislike가 증가하지 않았으면 숫자 증가. 증가중인 그룹에 같은 숫자 부여 (그룹화)
df_sort['dislikes_False'] = (df_sort['dislikes_up'] == False).cumsum()

# 두 값이 동등하게 나왔으나 알고보니 공동 1등 둘 있음
df_sort.groupby(['dislikes_False','title','channel_title']).size().sort_values(ascending=False).iloc[:5]
df_sort.groupby(['dislikes_False','channel_title'],as_index=False).size().sort_values(by='size',ascending=False).iloc[:5]

# 공동 1등 동시 추출
answerlist = df_sort.groupby(['dislikes_False', 'channel_title'], as_index=False).size().sort_values(by='size',ascending=False)
answer_max = answerlist.loc[answerlist['size'] >= answerlist.max().loc['size']]
print(answer_max['channel_title'].to_list())



# # 쓸모없는거 - as_index라는 좋은 물건을 몰랐을 떄나 하던 짓
# df_True = df_sort[df_sort['dislikes_up'] == True]
# df_True.groupby(['dislikes_False'])['channel_title']

# as_index 안쓸 시
# #index 사용법
# answer1 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).index[0][1]

# #idxmax, idxmin 사용법
# answer2 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).idxmax()[1]

# # index -> data로 변환
# answer3 = df_True.groupby(['title','channel_title']).size().sort_values(ascending=False).reset_index().iloc[0, 1]



#answer4 = df_True.groupby('')

#한줄 정리
# result = df_sort[df_sort['dislikes_up']].groupby(['channel_title', 'dislikes_False']).size().idxmax()[0]
# print(result)





['FiftyShadesVEVO', 'TWICE JAPAN OFFICIAL YouTube Channel']
       dislikes_False                         channel_title  size
30026           72630                       FiftyShadesVEVO    36
25851           63087  TWICE JAPAN OFFICIAL YouTube Channel    36
14849           35102                            KarolGVEVO    35
9416            23390                              Flo Rida    35
862              2603                               2CELLOS    35
...               ...                                   ...   ...
32670           82700                           AstronoGeek     1
32669           82699                           AstronoGeek     1
32668           82696                           AstronoGeek     1
32667           82695                           AstronoGeek     1
32683           82738                               Deja Vu     1

[32714 rows x 3 columns]


#### 결과적 풀이

In [ ]:
import pandas as pd

df = pd.read_csv('/data/48bd152a.csv')

df['trending_date2'] = pd.to_datetime(df['trending_date2'], format='%Y-%m-%d')
df_sort = df[['channelTitle', 'title','trending_date2','dislikes']].sort_values(by=['title','trending_date2'], ascending=True)
df_sort['dislikes_up'] = (df_sort.groupby(['title'])['trending_date2'].diff() == pd.Timedelta(1,'D'))&(df_sort.groupby(['channelTitle'])['trending_date2'].diff() == pd.Timedelta(1,'D')) & (df_sort.groupby(['title'])['dislikes'].diff() > 0 )


# 연속된 차수끼리 그룹화
df_sort['group'] = (df_sort['dislikes_up']== False).cumsum()
df_group = df_sort.groupby(['title','channelTitle','group'],as_index=False).size().sort_values('size', ascending=False)


# 문자열로 반환하려면 : 같은거 쓰지말고 좌표 정확하게 잡아서 출력할 것
answer = df_group.iloc[0,1]

print(answer)